<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_02_optimiser_comparison.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 02 — Adam, then L-BFGS

**Paired with L6.1 · Loss Functions and Gradients**

Every PINN in L7 to L12 is trained by **Adam first, then L-BFGS**. This notebook
measures why. You will

1. train a network with Adam and see where it slows down;
2. train the same network with L-BFGS from the start, and meet the closure it
   needs;
3. chain the two, Adam to get close and L-BFGS to finish, and measure what the
   handoff buys.

The target is **noiseless**, so the floor is zero and the differences between
the optimisers stay visible however long you train. Plain gradient descent,
random batches and momentum are taught in L6.1; this notebook keeps only the two
optimisers Part 2 uses.

## 0 · Setup

**What the three cells below do.** The first fetches the library file
`Ex_6_core.py` when you run on Colab. The second keeps what this notebook saves
in your Google Drive, so the report notebook can read it later. The third
imports the libraries and fixes the random seed.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
print("torch", torch.__version__, "| output dir:", core.OUTPUT_DIR)

## 1 · The problem

**What is fitted.** A damped oscillation, `exp(-0.9x) sin(4x)`: the way a
structure rings down after a knock. Two hundred samples of it, and a network
with two hidden layers of sixteen neurons. Small enough to train in seconds,
wiggly enough that the choice of optimiser shows.

**Why without noise.** With no noise the best possible loss is zero, so a lower
loss always means a better fit, and the gap between two optimisers is not
hidden under a noise floor. A PINN in Part 2 is in the same position: the
residual of an equation has no noise floor either.

In [ ]:
# The curve to fit: a damped oscillation, 200 noiseless samples.
#   core.response_dataset(n) -> x, y
#   core.to_tensor(a) makes the (N, 1) float32 column PyTorch wants
x, y = core.response_dataset(n=200)
X, Y = core.to_tensor(x), core.to_tensor(y)
loss_fn = nn.MSELoss()

core.set_seed(0)
print("parameters in the model:", core.count_parameters(core.MLP(hidden=(16, 16))))

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(x, y, lw=1.9, color="#1f77b4")
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title("The target: a damped structural response, sampled without noise")
ax.grid(alpha=0.25)
plt.show()

### Your turn

One function, used everywhere below. Keeping the seed reset **inside** it is
what makes the runs comparable: every optimiser starts from an identical
set of weights, so any difference you see is the optimiser and not the draw.

In [ ]:
# TODO: write run(make_opt, epochs) -> (history, model).
#
#   def run(make_opt, epochs=2000):
#       core.set_seed(0)                     # identical initial weights, always
#       model = core.MLP(hidden=(16, 16))
#       opt = make_opt(model.parameters())
#       history = []
#       for _ in range(epochs):
#           opt.zero_grad()
#           loss = loss_fn(model(X), Y)
#           loss.backward()
#           opt.step()
#           history.append(loss.item())
#       return np.array(history), model
#
# make_opt is a function of the parameters, not an optimiser object -- an
# optimiser is bound to the parameters of one model, so it cannot be reused
# across runs.

raise NotImplementedError("Write run(make_opt, epochs)")

## 2 · Adam

**What Adam does.** Plain gradient descent moves every parameter by the same
multiple of its gradient, so one learning rate has to suit both the steep and
the flat directions of the loss. Adam keeps two running averages for each
parameter: the average gradient, which smooths the direction as momentum does,
and the average squared gradient, which sets the size of that parameter's step.
Every parameter then moves by about the learning rate, whatever the scale of
its gradient. That is why one learning rate works for most problems, and why
Adam copes well far from the answer.

**What to look for.** How fast the loss falls at first, and how slowly it falls
at the end. Near the minimum Adam's steps stay about the learning rate in size,
so it circles the bottom rather than settling into it.

In [ ]:
# Adam, the first stage of the Part 2 recipe: lr 0.01, 2000 steps, each
# step on all 200 samples. run() resets the seed, so every run in this
# notebook starts from the same weights.
histories = {}
histories["Adam"], adam_model = run(lambda p: torch.optim.Adam(p, lr=0.01), epochs=2000)
print(f"Adam, 2000 steps: final loss {histories['Adam'][-1]:.2e}")

core.plot_curves(histories, title="Adam: a fast start, a slow finish")
plt.show()

**What you should see.** A final loss of about $2 \times 10^{-5}$, and a curve
that falls by orders of magnitude in the first few hundred steps, then flattens
with small spikes. The spikes are Adam's steps overshooting the bottom of the
loss, which is why its loss is not monotone.

That flat, spiky end is what L-BFGS is for.

## 3 · L-BFGS, and why its interface is different

**What L-BFGS does.** Adam uses only the slope of the loss. L-BFGS
(limited-memory Broyden–Fletcher–Goldfarb–Shanno) also estimates its
**curvature**, how the slope changes, from the last few gradients. With the
curvature it can choose both the direction and the length of a step, as
Newton's method does, and near a minimum that converges far faster than steps
of a fixed size.

**What to look for.** The cell below runs L-BFGS from the same starting weights
as Adam, with no Adam before it. Compare its final loss with Adam's.

L-BFGS estimates the curvature from the last few gradients, chooses a direction
and a distance, and runs a line search, **evaluating the loss several times per
step**. So it needs a *closure*, a function that does the full zero-grad,
forward and backward each time it is called. It is **full batch**, because the
line search compares losses, and it is fast near a minimum and unreliable far
from one: the opposite of Adam.

In [ ]:
# L-BFGS is a different animal: it uses curvature and needs a
# closure that re-evaluates the loss, because it may call it
# several times per step.
core.set_seed(0)
lbfgs_model = core.MLP(hidden=(16, 16))
opt = torch.optim.LBFGS(lbfgs_model.parameters(), lr=1.0, max_iter=20)

def closure():
    opt.zero_grad()
    loss = loss_fn(lbfgs_model(X), Y)
    loss.backward()
    return loss

lbfgs_history = []
for _ in range(60):
    opt.step(closure)
    with torch.no_grad():
        lbfgs_history.append(loss_fn(lbfgs_model(X), Y).item())

print(f"L-BFGS from a cold start, 60 steps: {lbfgs_history[-1]:.8f}")

## 4 · The handoff Part 2 relies on

**Why chain them.** Adam is robust far from the answer and slow to polish.
L-BFGS is the reverse: fast near a minimum, unreliable far from one. So run Adam
until its loss curve flattens, then hand the same network to L-BFGS to finish:
**Adam to get close, L-BFGS to finish**. This is the recipe of every PINN
exercise in Part 2.

**The comparison.** Four recipes, all from the same starting weights: Adam
alone, L-BFGS alone, and Adam for 1000 or for 2000 steps followed by 60 L-BFGS
steps on the same model. The two chained runs show whether a longer Adam stage
pays.

### Your turn

In [ ]:
# TODO: chain the two, and compare against equal-budget alternatives.
#
#   Build FOUR results, all starting from core.set_seed(0) and core.MLP((16,16)):
#
#     "Adam 2000"            Adam(lr=0.01), 2000 epochs
#     "L-BFGS 60"            L-BFGS as in section 3, 60 steps, from cold
#     "Adam 1000 -> L-BFGS"  Adam(lr=0.01) for 1000 epochs, then L-BFGS on the
#                            SAME model for 60 steps
#     "Adam 2000 -> L-BFGS"  Adam for 2000 epochs, then L-BFGS for 60 steps
#
#   Record the final loss of each in final = {name: loss}.
#
#   For the chained runs, build the L-BFGS optimiser AFTER Adam has finished,
#   over the same model.parameters(). A closure closes over the model, so
#   define it after the model exists.

raise NotImplementedError("Chain Adam into L-BFGS")

In [ ]:
# Adam first, then L-BFGS to finish - against the same budget spent
# entirely on one or the other. This pairing is what the PINN
# notebooks in Part 2 use.
print(core.error_table(
    [[name, f"{loss:.3e}"] for name, loss in final.items()],
    ["recipe", "final training loss"]))

**What you should see.** The chained runs reach a loss several orders of
magnitude below Adam alone, and cold L-BFGS lands somewhere unimpressive: it is
a local method handed a bad starting point. That is the justification for the
two-stage training of Part 2.

A caution for Part 2: a loss of `1e-8` on a PINN residual says the network
satisfies the equations you wrote at the points you sampled, not that the
solution is right. Convergence is not correctness.

**In practice.** Hand over when Adam's loss curve has flattened, not at a fixed
step count, and check that L-BFGS lowered the loss. If it stops after one or two
steps, or the loss jumps, the handoff has gone wrong: usually a step too large
for L-BFGS, or a loss that changes between calls, from random batches or
resampled points, which its comparisons cannot handle.

## 5 · What this notebook does not show

* **No SGD, momentum or learning-rate sweep.** L6.1 teaches them; this notebook
  keeps the two optimisers Part 2 uses. Every step here is on all two hundred
  samples.
* **No line search in L-BFGS.** Part 2 builds it with
  `line_search_fn="strong_wolfe"`, which makes it more reliable, most of all
  from a cold start.
* **No generalisation.** Training loss only, on noiseless data.
* **One problem, one architecture, one seed.** The ordering is typical, not
  universal.

## 6 · Save

**What this does.** It writes the Adam and L-BFGS loss curves and the four
recipes' final losses to a file that the report in notebook 05 reads. There is
nothing to change here.

In [ ]:
# Saved for the report in notebook 05.
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb02_optimisers.npz")
np.savez(path,
         hist_adam=histories["Adam"],
         hist_lbfgs=np.asarray(lbfgs_history),
         final_names=np.array(list(final)),
         final_losses=np.asarray(list(final.values()), dtype=float))
print("wrote", path)
core.saved(path)


## 7 · Before you move on

You should be able to answer these without rerunning anything. Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Adam reached a loss of about $2 \times 10^{-5}$ and then fell only slowly,
   with small spikes. What two running averages does Adam keep, what does each
   fix compared with plain gradient descent, and why is its loss not monotone
   near the minimum?
   *→ L6.1 Q9*
2. Every exercise in Part 2 trains with Adam first, then L-BFGS. How would you
   decide when to hand over, and what would you see in the loss if the handoff
   went wrong?
   *→ L6.1 Q9*
3. Why does L-BFGS need a closure when Adam does not? Say what L-BFGS uses that
   Adam does not, why it has to be full-batch, and what each call of the closure
   costs in forward and backward passes.
   *→ L6.1 Q7, Q9*
4. Why is cold L-BFGS worse than Adam-then-L-BFGS, given that L-BFGS uses more
   information per step? When is the hand-off worth it?
   *→ L6.1 Q9*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 03](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_03_transfer_and_fine_tuning.ipynb)**, where a trained network meets a second machine and most
of what it learned turns out to still be useful.